# Day 12：网络请求与 API 调用（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 2 周**：第 2 周：面向对象、工程化与命令行 AI 助手实战  
> **GitLab 工单**：`STARMIND-012` ｜ **开发分支**：`feat/day12-http-llm-api-client`  
> **配套 HTML5 课件**：`courseware/day12/index.html` ｜ **独立版本快照**：`versions/day12/`

---

## 🎙️ 老陈开场旁白

兄弟们，今天是个里程碑！前 11 天咱们都在本地捣鼓数据结构、类和文件，今天终于要「通网」调用大模型 API 了（工单 STARMIND-012）！
先说一个国内 AI 工程界的超级利好：不管是 DeepSeek、阿里通义千问（DashScope）、智谱 GLM 还是硅基流动，几乎全部兼容 OpenAI 的 `/v1/chat/completions` HTTP 接口规范——只要你吃透 HTTP POST 请求、Bearer Token 鉴权和 JSON 报文解析，换一家厂商只需要改两行配置（`base_url` 和 `model`）！
老陈还要教你一个大厂极其看重的习惯：**无 Key 自动降级本地 Mock 模式**。开发机没配外网 Key、或者 GitLab CI 跑自动化测试时，绝不能因为连不上外网就报错挂掉，更不能在单元测试里白白烧公司的 Token 钱！

### 承上启下
- **昨日基线**：昨天咱们用 `pathlib`、`re`、`csv`、`datetime` 撸完了星辰知识库批量文档扫描与报表导出工具，本地文件处理这一关已经彻底打通。
- **明日预告**：明天（Day 13）咱们进入 Python 工程化进阶语法——装饰器、生成器 `yield`、`python-dotenv` 密钥管理与 `asyncio` 异步并发，给今天刚调通的 API 客户端加上自动重试、超时控制与流式输出！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day12'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-012）

星辰智脑项目需要正式打通大模型推理能力，首期计划接入 DeepSeek 与通义千问（OpenAI 兼容接口）。但在内网开发和自动化测试环境下无法保证随时拥有外网 API Key，需要客户端具备平滑的本地 Mock 降级能力。

**今日迭代目标：**
- 理解 HTTP 协议核心要素（URL、POST Method、Headers、JSON Body、状态码 200/401/429/5xx）。
- 掌握 `httpx` / `urllib` / `requests` 的区别与 OpenAI 兼容协议 `/chat/completions` 报文封装。
- 交付具备「真实 HTTP 调用 + 无 Key 自动本地 Mock 降级」的客户端类与第一版命令行 AI 问答小程序。

**验收标准：**
- 当 `api_key` 为空或开启 `mock_mode=True` 时，自动返回符合 OpenAI 协议格式的本地 Mock 回复且不发起任何外网请求。
- 当传入 `api_key` 时，能按标准格式组装 `Authorization: Bearer ...` 与 JSON 请求体并发送 POST 请求，正确解析 `choices[0].message.content` 与 `usage`。
- 能准确拦截 401 鉴权失败、429 触发限流、5xx 服务端异常，并给出清晰中文错误提示；通过 `httpx.MockTransport` 离线单测。

---

## 📚 课堂核心技术笔记

### 1. 1. HTTP 协议核心：调用大模型 API 必懂的 4 个要素

调用大模型本质上就是发一次标准 HTTP POST 请求：
- **URL**：如 DeepSeek 的 `https://api.deepseek.com/v1/chat/completions` 或通义千问兼容地址 `https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions`。
- **Headers**：必须携带 `Authorization: Bearer <API_KEY>` 与 `Content-Type: application/json`。
- **Body**：JSON 对象，包含 `model`、`messages`（角色+内容列表）、`temperature` 等参数。
- **状态码**：`200` 成功、`401` Key 无效或过期、`429` 触发 QPS 限流或余额不足、`500/502/503` 模型服务端抖动。

> **⚠️ 老陈避坑**：调用外部 HTTP API 一定要显式传 `timeout=30.0`！默认不设超时一旦对端网络黑洞，你的服务线程就会永远卡死直到被运维重启。

### 2. 2. urllib vs requests vs httpx：现代 AI 工程怎么选？

- `urllib.request`：Python 标准库，零第三方依赖，写轻量脚本够用，但 API 略繁琐。
- `requests`：老牌国民级同步 HTTP 库，写法优雅，但**不支持 `async/await` 异步和 HTTP/2**。
- `httpx`：新一代主力库，接口几乎 100% 兼容 `requests`，同时原生支持同步 `Client`、异步 `AsyncClient` 以及 `MockTransport` 离线测试。OpenAI 官方 Python SDK 和 FastAPI 测试客户端底层全用 `httpx`！

> **⚠️ 老陈避坑**：高频调用同一个 API 域名时，不要每次请求都临时新建客户端，尽量复用 `httpx.Client()` 连接池（Keep-Alive），能省掉大量 TCP+TLS 握手耗时。

### 3. 3. 为什么企业级 AI 客户端必须内置 Mock Fallback？

1）新同学入职第一天还没审批下来正式 API Key，也能立刻跑通全套业务代码；
2）CI/CD 自动化测试每天跑几百次单测，用本地 Mock 或 `httpx.MockTransport` 可以实现零成本、毫秒级、100% 确定性通过，不受外网波动影响。

> **⚠️ 老陈避坑**：Mock 数据结构必须严格对齐真实 API 的返回 Schema（包含 `choices` 和 `usage`），让 Mock 和真实请求走同一套解析函数，否则「测试全绿、上线解析报错」。


## 💻 实操步骤 1：步骤一：封装 OpenAI 兼容 HTTP 大模型客户端（含本地 Mock 降级）

- **目标文件**：`versions/day12/src/day12_llm_http_client.py`
- **老陈代码旁白**：先写核心客户端 `day12_llm_http_client.py`：用 `dataclass` 定义配置与响应，纯函数负责报文构造与解析，`StarMindHTTPClient` 在无 Key 时无缝降级为本地 Mock，有 Key 时走标准 `httpx` POST 请求。


In [ ]:
"""
STARMIND-012 模块一：OpenAI 兼容协议 HTTP 大模型客户端（支持本地 Mock 降级）。
兼容 DeepSeek / 通义千问 DashScope / 智谱等 /v1/chat/completions 接口。
"""

import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import httpx


class LLMAPIError(RuntimeError):
    """大模型 HTTP 接口调用异常基类，携带 HTTP 状态码便于上层区分处理。"""

    def __init__(self, message: str, status_code: Optional[int] = None) -> None:
        super().__init__(message)
        self.status_code = status_code


@dataclass
class LLMRequestConfig:
    """大模型 API 连接配置。"""

    base_url: str = "https://api.deepseek.com/v1"
    api_key: str = ""
    model: str = "deepseek-chat"
    timeout: float = 15.0
    temperature: float = 0.7


@dataclass
class LLMResponse:
    """标准化大模型响应对象。"""

    content: str
    model: str
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int
    latency_ms: float
    source: str  # "remote" 或 "mock"


def build_chat_payload(
    messages: List[Dict[str, str]],
    model: str,
    temperature: float = 0.7,
) -> Dict[str, Any]:
    """构造标准 OpenAI 兼容 /chat/completions 请求体。"""
    if not messages:
        raise ValueError("messages 不能为空")
    return {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "stream": False,
    }


def parse_chat_response(
    raw_data: Dict[str, Any],
    latency_ms: float,
    source: str = "remote",
) -> LLMResponse:
    """解析 OpenAI 兼容响应 JSON 为强类型 LLMResponse。"""
    choices = raw_data.get("choices")
    if not isinstance(choices, list) or not choices:
        raise ValueError("接口返回格式异常：缺少有效的 choices 字段")

    first_message = choices[0].get("message", {})
    content = first_message.get("content")
    if not isinstance(content, str):
        raise ValueError("接口返回格式异常：缺少 message.content 文本")

    usage = raw_data.get("usage", {})
    prompt_tokens = int(usage.get("prompt_tokens", 0))
    completion_tokens = int(usage.get("completion_tokens", 0))
    total_tokens = int(usage.get("total_tokens", prompt_tokens + completion_tokens))

    return LLMResponse(
        content=content.strip(),
        model=str(raw_data.get("model", "unknown")),
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=total_tokens,
        latency_ms=round(latency_ms, 2),
        source=source,
    )


class StarMindHTTPClient:
    """星辰智脑 OpenAI 兼容 HTTP 客户端（支持无 Key 本地 Mock 降级）。"""

    def __init__(
        self,
        config: Optional[LLMRequestConfig] = None,
        mock_mode: bool = False,
        transport: Optional[httpx.BaseTransport] = None,
    ) -> None:
        self.config = config or LLMRequestConfig()
        # 如果显式开启 mock_mode 或者未配置 api_key，自动启用本地 Mock 模式
        self.mock_mode = mock_mode or (not self.config.api_key.strip())
        self._transport = transport

    def _mock_chat_completion(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """离线本地 Mock 引擎：返回与真实 OpenAI 协议 100% 同构的 JSON 报文。"""
        messages = payload["messages"]
        last_user = next(
            (m["content"] for m in reversed(messages) if m.get("role") == "user"),
            "",
        )

        # 针对星辰数科常见演示问题给出贴近业务的回答
        if "报销" in last_user:
            reply = "【本地Mock】星辰数科差旅报销请在钉钉发起「费用报销单」，500元以上需直属主管+财务双审批。"
        elif "DeepSeek" in last_user or "兼容" in last_user or "通义" in last_user:
            reply = "【本地Mock】DeepSeek 与通义千问均兼容 OpenAI /v1/chat/completions 协议，切换厂商只需修改 base_url 与 model。"
        elif "年假" in last_user:
            reply = "【本地Mock】入职满 1 年享 5 天带薪年假，系统按自然年每年 1 月 1 日自动发放。"
        else:
            reply = f"【本地Mock】已收到你的提问：「{last_user}」。当前未配置线上 API Key，已自动降级为离线模拟回复。"

        prompt_tokens = sum(len(m.get("content", "")) for m in messages)
        completion_tokens = len(reply)
        return {
            "id": "chatcmpl-starmind-mock-001",
            "object": "chat.completion",
            "model": f"{payload['model']}-mock",
            "choices": [
                {
                    "index": 0,
                    "message": {"role": "assistant", "content": reply},
                    "finish_reason": "stop",
                }
            ],
            "usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
            },
        }

    def chat(self, messages: List[Dict[str, str]]) -> LLMResponse:
        """发送对话请求并返回标准化 LLMResponse。"""
        payload = build_chat_payload(
            messages=messages,
            model=self.config.model,
            temperature=self.config.temperature,
        )
        start = time.perf_counter()

        if self.mock_mode:
            raw_data = self._mock_chat_completion(payload)
            elapsed_ms = (time.perf_counter() - start) * 1000
            return parse_chat_response(raw_data, latency_ms=elapsed_ms, source="mock")

        endpoint = f"{self.config.base_url.rstrip('/')}/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.config.api_key}",
            "Content-Type": "application/json",
        }

        try:
            with httpx.Client(
                timeout=self.config.timeout,
                transport=self._transport,
            ) as client:
                resp = client.post(endpoint, headers=headers, json=payload)
        except httpx.TimeoutException as exc:
            raise LLMAPIError(f"请求大模型超时（>{self.config.timeout}s）") from exc
        except httpx.HTTPError as exc:
            raise LLMAPIError(f"网络请求失败：{exc}") from exc

        if resp.status_code == 401:
            raise LLMAPIError("API Key 无效或已过期（HTTP 401）", status_code=401)
        if resp.status_code == 429:
            raise LLMAPIError("触发接口频率限制或余额不足（HTTP 429）", status_code=429)
        if resp.status_code >= 500:
            raise LLMAPIError(
                f"大模型服务端暂时不可用（HTTP {resp.status_code}）",
                status_code=resp.status_code,
            )
        if resp.status_code != 200:
            raise LLMAPIError(
                f"接口返回异常状态码 HTTP {resp.status_code}: {resp.text[:120]}",
                status_code=resp.status_code,
            )

        elapsed_ms = (time.perf_counter() - start) * 1000
        return parse_chat_response(resp.json(), latency_ms=elapsed_ms, source="remote")


if __name__ == "__main__":
    # 默认不传 api_key，自动触发本地 Mock 降级模式
    client = StarMindHTTPClient(LLMRequestConfig(model="deepseek-chat"))
    result = client.chat([
        {"role": "system", "content": "你是杭州星辰数科企业知识助手。"},
        {"role": "user", "content": "星辰数科差旅报销怎么走流程？"},
    ])
    print(f"[{result.source}] 模型: {result.model} | 耗时: {result.latency_ms}ms | Tokens: {result.total_tokens}")
    print(f"回复: {result.content}")


### 🔍 步骤 1 代码深度解析

1. `self.mock_mode = mock_mode or (not self.config.api_key.strip())` 实现了极度丝滑的无 Key 自动降级，开发者零配置即可跑通。
2. 无论走 `mock` 还是 `remote`，最终都经过 `parse_chat_response()` 统一解析，保证数据结构 100% 一致。
3. 针对 HTTP 401/429/5xx 做了细粒度状态码翻译，线上排障一眼就能定位是 Key 错、限流还是厂商挂了。

**预期运行输出参考：**
```text
[mock] 模型: deepseek-chat-mock | 耗时: 0.05ms | Tokens: 78
回复: 【本地Mock】星辰数科差旅报销请在钉钉发起「费用报销单」，500元以上需直属主管+财务双审批。
```


## 💻 实操步骤 2：步骤二：组装第一版命令行 AI 问答小程序（支持脚本化非阻塞测试）

- **目标文件**：`versions/day12/src/day12_cli_qa.py`
- **老陈代码旁白**：客户端就绪后，咱们把它包装成一个干净的命令行 AI 问答小程序。注意老陈的工程铁律：交互循环必须支持注入测试输入列表 `inputs`，绝对不能让代码一导入或一测试就卡死在 `input()` 里！


In [ ]:
"""
STARMIND-012 模块二：星辰智脑命令行 AI 问答小程序 v0.1。
演示单轮提问封装、控制台格式化输出与可测试的 CLI 交互循环。
"""

from typing import List, Optional

from day12_llm_http_client import LLMAPIError, LLMRequestConfig, LLMResponse, StarMindHTTPClient

DEFAULT_SYSTEM_PROMPT = "你是杭州星辰数科内部 AI 助手「星辰智脑」，请用简洁专业中文回答问题。"


def ask_once(
    client: StarMindHTTPClient,
    question: str,
    system_prompt: str = DEFAULT_SYSTEM_PROMPT,
) -> LLMResponse:
    """发起一次单轮 AI 问答。"""
    cleaned = question.strip()
    if not cleaned:
        raise ValueError("提问内容不能为空")

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": cleaned},
    ]
    return client.chat(messages)


def format_qa_panel(question: str, response: LLMResponse) -> str:
    """将问答结果格式化为清晰的终端展示文本。"""
    return (
        f"问：{question}\n"
        f"答：{response.content}\n"
        f"—— [来源: {response.source} | 模型: {response.model} | "
        f"Tokens: {response.total_tokens} | 耗时: {response.latency_ms}ms]"
    )


def run_cli_qa(
    client: StarMindHTTPClient,
    inputs: Optional[List[str]] = None,
) -> List[str]:
    """
    运行命令行问答循环。
    - 传入 inputs 列表时按脚本逐条执行（适合演示与自动化测试，不阻塞）；
    - inputs 为 None 时进入终端交互式 input() 模式。
    """
    outputs: List[str] = []
    script_iter = iter(inputs) if inputs is not None else None

    while True:
        if script_iter is not None:
            try:
                raw_q = next(script_iter)
            except StopIteration:
                break
        else:
            raw_q = input("请输入问题（输入 /exit 退出）> ")

        q = raw_q.strip()
        if q.lower() in {"/exit", "exit", "quit"}:
            outputs.append("已退出星辰智脑问答小程序。")
            break
        if not q:
            outputs.append("提示：问题不能为空，请重新输入。")
            continue

        try:
            resp = ask_once(client, q)
            outputs.append(format_qa_panel(q, resp))
        except (ValueError, LLMAPIError) as exc:
            outputs.append(f"调用失败：{exc}")

    return outputs


if __name__ == "__main__":
    # 使用预设问题脚本运行演示，无需手动按键输入
    demo_client = StarMindHTTPClient(LLMRequestConfig(model="deepseek-chat"))
    demo_questions = [
        "星辰数科差旅报销流程是什么？",
        "DeepSeek API 兼容什么协议格式？",
        "/exit",
    ]
    for block in run_cli_qa(demo_client, inputs=demo_questions):
        print(block)
        print("-" * 50)


### 🔍 步骤 2 代码深度解析

1. `run_cli_qa(client, inputs=...)` 把「数据来源」抽象出来，传入 `inputs` 列表就能全自动跑完多条问答并收集输出，既方便演示又方便 `pytest` 断言。
2. 在循环内捕获 `(ValueError, LLMAPIError)`，哪怕某一次请求遇到 429 限流或空输入，程序也只会给出友好提示而不会闪退崩溃。

**预期运行输出参考：**
```text
问：星辰数科差旅报销流程是什么？
答：【本地Mock】星辰数科差旅报销请在钉钉发起「费用报销单」，500元以上需直属主管+财务双审批。
—— [来源: mock | 模型: deepseek-chat-mock | Tokens: 96 | 耗时: 0.04ms]
--------------------------------------------------
问：DeepSeek API 兼容什么协议格式？
答：【本地Mock】DeepSeek 与通义千问均兼容 OpenAI /v1/chat/completions 协议，切换厂商只需修改 base_url 与 model。
—— [来源: mock | 模型: deepseek-chat-mock | Tokens: 109 | 耗时: 0.03ms]
--------------------------------------------------
已退出星辰智脑问答小程序。
--------------------------------------------------
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day12/tests/test_day12.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-012 自动化单元测试（全程离线，零外网依赖）。"""

import sys
from pathlib import Path
import httpx
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day12_llm_http_client import (
    LLMAPIError,
    LLMRequestConfig,
    StarMindHTTPClient,
    build_chat_payload,
    parse_chat_response,
)
from day12_cli_qa import ask_once, run_cli_qa


def test_payload_and_response_parser():
    payload = build_chat_payload([{"role": "user", "content": "你好"}], model="qwen-plus", temperature=0.3)
    assert payload["model"] == "qwen-plus"
    assert payload["temperature"] == 0.3

    raw = {
        "model": "qwen-plus",
        "choices": [{"message": {"role": "assistant", "content": "  你好，星辰同事！  "}}],
        "usage": {"prompt_tokens": 5, "completion_tokens": 10, "total_tokens": 15},
    }
    resp = parse_chat_response(raw, latency_ms=12.345, source="remote")
    assert resp.content == "你好，星辰同事！"
    assert resp.total_tokens == 15
    assert resp.latency_ms == 12.35

    with pytest.raises(ValueError):
        parse_chat_response({"choices": []}, latency_ms=1.0)


def test_auto_mock_fallback_without_api_key():
    client = StarMindHTTPClient(LLMRequestConfig(api_key="", model="deepseek-chat"))
    assert client.mock_mode is True
    resp = ask_once(client, "年假有多少天？")
    assert resp.source == "mock"
    assert "年假" in resp.content


def test_remote_http_success_and_error_codes_with_mock_transport():
    # 1. 模拟 HTTP 200 正常返回并校验 Authorization 请求头
    def ok_handler(request: httpx.Request) -> httpx.Response:
        assert request.headers["Authorization"] == "Bearer sk-test-key"
        assert request.url.path.endswith("/chat/completions")
        return httpx.Response(
            200,
            json={
                "model": "deepseek-chat",
                "choices": [{"message": {"role": "assistant", "content": "来自远程模拟节点的回复"}}],
                "usage": {"prompt_tokens": 12, "completion_tokens": 18, "total_tokens": 30},
            },
        )

    cfg = LLMRequestConfig(api_key="sk-test-key", model="deepseek-chat")
    client_ok = StarMindHTTPClient(cfg, transport=httpx.MockTransport(ok_handler))
    resp = client_ok.chat([{"role": "user", "content": "测试远程调用"}])
    assert resp.source == "remote"
    assert resp.content == "来自远程模拟节点的回复"
    assert resp.total_tokens == 30

    # 2. 模拟 HTTP 401 鉴权失败与 HTTP 429 限流
    client_401 = StarMindHTTPClient(
        cfg,
        transport=httpx.MockTransport(lambda req: httpx.Response(401, json={"error": "invalid key"})),
    )
    with pytest.raises(LLMAPIError) as exc_401:
        client_401.chat([{"role": "user", "content": "hi"}])
    assert exc_401.value.status_code == 401

    client_429 = StarMindHTTPClient(
        cfg,
        transport=httpx.MockTransport(lambda req: httpx.Response(429, json={"error": "rate limit"})),
    )
    with pytest.raises(LLMAPIError) as exc_429:
        client_429.chat([{"role": "user", "content": "hi"}])
    assert exc_429.value.status_code == 429


def test_run_cli_qa_scripted():
    client = StarMindHTTPClient(LLMRequestConfig(api_key=""))
    logs = run_cli_qa(client, inputs=["", "报销流程", "/exit"])
    assert "不能为空" in logs[0]
    assert "费用报销单" in logs[1]
    assert "已退出" in logs[2]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：HTTP 状态码智能诊断器

**题目要求：**
编写函数 `diagnose_http_status(status_code: int) -> dict`，根据输入的状态码返回 `{'category': ..., 'retryable': bool, 'advice': ...}`。要求正确区分 200（成功，不可重试）、401/403（鉴权错误，不可重试）、429（限流，可重试）、500~599（服务端错误，可重试）。

**思路提示：** 用范围判断 `200 <= status_code < 300` 和 `500 <= status_code < 600` 处理类别，429 单独标为 `retryable=True`。


In [ ]:
"""Day 12 课后题 1：大模型 API HTTP 状态码智能诊断器。"""

from typing import Any, Dict


def diagnose_http_status(status_code: int) -> Dict[str, Any]:
    """根据 HTTP 状态码返回故障分类、是否建议重试及排查建议。"""
    if 200 <= status_code < 300:
        return {"category": "success", "retryable": False, "advice": "请求成功，正常解析响应即可。"}
    if status_code in (401, 403):
        return {"category": "auth_error", "retryable": False, "advice": "检查 API Key 是否正确、过期或缺少权限。"}
    if status_code == 429:
        return {"category": "rate_limit", "retryable": True, "advice": "触发限流或配额不足，建议指数退避重试或检查账户余额。"}
    if 500 <= status_code < 600:
        return {"category": "server_error", "retryable": True, "advice": "大模型厂商服务端抖动，建议稍后重试或切换备用模型。"}
    return {"category": "client_error", "retryable": False, "advice": f"客户端请求异常（HTTP {status_code}），请检查 URL 与请求体格式。"}


if __name__ == "__main__":
    for code in (200, 401, 429, 503):
        print(code, "->", diagnose_http_status(code))


**答案解析与评分标准：**

区分「可重试错误（429/5xx）」和「不可重试错误（400/401/403）」是写好重试中间件的第一步——对着 401 疯狂重试只会浪费算力甚至被封 IP。


### 作业 2（企业实战）：企业实战：国内多厂商 OpenAI 兼容配置工厂

**题目要求：**
编写 `create_provider_config(provider: str, api_key: str) -> dict`，支持 `'deepseek'`、`'qwen'`（通义千问）、`'zhipu'`（智谱）三种国内主流厂商，自动填入对应的兼容 `base_url` 与默认推荐模型名；遇到未知厂商抛出 `ValueError`。

**思路提示：** 定义一个 `PROVIDER_PRESETS` 常量字典，对传入的 `provider.strip().lower()` 做查表校验。


In [ ]:
"""Day 12 课后题 2：国内多厂商 OpenAI 兼容配置工厂。"""

from typing import Dict

PROVIDER_PRESETS: Dict[str, Dict[str, str]] = {
    "deepseek": {
        "base_url": "https://api.deepseek.com/v1",
        "model": "deepseek-chat",
    },
    "qwen": {
        "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
        "model": "qwen-plus",
    },
    "zhipu": {
        "base_url": "https://open.bigmodel.cn/api/paas/v4",
        "model": "glm-4-flash",
    },
}


def create_provider_config(provider: str, api_key: str) -> Dict[str, str]:
    """根据厂商名称生成标准 OpenAI 兼容客户端配置。"""
    key = provider.strip().lower()
    if key not in PROVIDER_PRESETS:
        supported = ", ".join(sorted(PROVIDER_PRESETS))
        raise ValueError(f"不支持的厂商 '{provider}'，当前仅支持: {supported}")
    preset = PROVIDER_PRESETS[key]
    return {
        "provider": key,
        "base_url": preset["base_url"],
        "model": preset["model"],
        "api_key": api_key,
    }


if __name__ == "__main__":
    print(create_provider_config("DeepSeek", "sk-demo"))
    print(create_provider_config("qwen", "sk-demo"))


**答案解析与评分标准：**

利用 OpenAI 协议兼容特性，通过工厂字典集中管理各家 `base_url` 和默认模型，业务代码零耦合，切换供应商一行配置搞定。


### 作业 3（架构进阶）：架构进阶：带自定义 Trace-ID 的 httpx 拦截测试封装

**题目要求：**
编写函数 `send_traced_request(url: str, trace_id: str, transport: httpx.BaseTransport) -> dict`，发送携带 `X-Trace-Id` 请求头的 GET 请求，校验响应 JSON 中是否正确回显了该 `trace_id`，并用 `httpx.MockTransport` 编写自包含运行验证。

**思路提示：** 在 `httpx.MockTransport` 的回调函数里从 `request.headers.get('X-Trace-Id')` 取出追踪号并放入返回 JSON。


In [ ]:
"""Day 12 课后题 3：带 X-Trace-Id 链路追踪头的 httpx 请求与离线 Mock 验证。"""

from typing import Any, Dict
import httpx


def send_traced_request(
    url: str,
    trace_id: str,
    transport: httpx.BaseTransport,
) -> Dict[str, Any]:
    """发送带链路追踪 ID 的 GET 请求并校验回显。"""
    with httpx.Client(timeout=5.0, transport=transport) as client:
        resp = client.get(url, headers={"X-Trace-Id": trace_id})
        resp.raise_for_status()
        data = resp.json()
    if data.get("echo_trace_id") != trace_id:
        raise ValueError("服务端未正确回显 X-Trace-Id")
    return data


if __name__ == "__main__":
    def mock_server(request: httpx.Request) -> httpx.Response:
        return httpx.Response(
            200,
            json={"status": "ok", "echo_trace_id": request.headers.get("X-Trace-Id")},
        )

    result = send_traced_request(
        "https://api.starmind.local/health",
        trace_id="trace-20260512-001",
        transport=httpx.MockTransport(mock_server),
    )
    print("链路追踪验证通过:", result)


**答案解析与评分标准：**

在微服务和 AI 网关中，每次请求携带 `X-Trace-Id` 是排查全链路耗时的命根子，而 `httpx.MockTransport` 让我们无需起真实服务器就能在毫秒级验证请求头透传逻辑。


---

## 🎯 老陈收尾复盘

今天你不仅搞懂了 HTTP 请求头、状态码和 JSON 报文，还亲手写出了一套「有 Key 走真实 HTTPS、无 Key 自动走本地 Mock、单测用 `httpx.MockTransport` 精准模拟 200/401/429」的生产级客户端骨架。
记住老陈一句话：**永远不要把真实 API Key 硬编码写进 `.py` 文件里提交到 Git！**明天咱们就用 `python-dotenv` + 重试装饰器 + 异步并发，把这套客户端武装到牙齿！
